# 03 Analysis and Insights
Headline numbers, delivery and review findings, category concentration, and recommendations.

In [1]:
# setup
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.utils import load_orders_clean, get_engine

df = load_orders_clean()
df["year"] = df["order_purchase_timestamp"].dt.year
df["month_num"] = df["order_purchase_timestamp"].dt.month

In [2]:
# Headlines Numbers
total_rev = df["items_value"].sum()
orders = len(df)
customers = df["customer_unique_id"].nunique()
repeat = (df.groupby("customer_unique_id").size() > 1).mean() * 100

# Fair growth comparison: Jan-Aug of each year
ytd = df[df["month_num"] <= 8].groupby("year")["items_value"].sum()
yoy = (ytd[2018] / ytd[2017] - 1) * 100

print(f"Orders: {orders:,} | Customers: {customers:,}")
print(f"Revenue (items): {total_rev:,.0f} BRL")
print(f"Jan-Aug 2018 vs Jan-Aug 2017 revenue growth: {yoy:.1f}%")
print(f"Repeat customer rate: {repeat:.2f}%")

Orders: 96,470 | Customers: 93,350
Revenue (items): 13,220,249 BRL
Jan-Aug 2018 vs Jan-Aug 2017 revenue growth: 141.1%
Repeat customer rate: 3.00%


In [3]:
# delivery and reviews finding 
rv = df.dropna(subset=["review_score"])
late_avg = rv.loc[rv["is_late"] == 1, "review_score"].mean()
ontime_avg = rv.loc[rv["is_late"] == 0, "review_score"].mean()
very_late_avg = rv.loc[rv["delay_days"] > 7, "review_score"].mean()

print(f"Late share: {df['is_late'].mean() * 100:.1f}%")
print(f"Avg review: on time {ontime_avg:.2f} vs late {late_avg:.2f} vs 8+ days late {very_late_avg:.2f}")

worst = (df.groupby("customer_state")
           .agg(orders=("order_id", "count"),
                avg_delivery=("delivery_days", "mean"),
                late_pct=("is_late", lambda s: s.mean() * 100))
           .query("orders >= 200").sort_values("late_pct", ascending=False).head(5).round(1))
print(worst)

Late share: 6.8%
Avg review: on time 4.29 vs late 2.27 vs 8+ days late 1.70
                orders  avg_delivery  late_pct
customer_state                                
AL                 397          24.0      21.4
MA                 717          21.1      17.4
SE                 335          21.0      15.2
PI                 476          19.0      13.9
CE                1279          20.8      13.8


In [4]:
# Category concentration
cat = pd.read_sql("""
    SELECT COALESCE(t.product_category_name_english, 'unknown') AS category,
           SUM(oi.price) AS revenue
    FROM order_items oi
    JOIN orders_clean o ON oi.order_id = o.order_id
    JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
    GROUP BY category ORDER BY revenue DESC
""", get_engine())
top3 = cat["revenue"].head(3).sum() / cat["revenue"].sum() * 100
print(cat.head(3)); print(f"Top 3 categories = {top3:.1f}% of revenue")

         category     revenue
0   health_beauty  1233131.72
1   watches_gifts  1165898.98
2  bed_bath_table  1023434.76
Top 3 categories = 25.9% of revenue


## Findings
1. Jan-Aug 2018 revenue was 141.1% above Jan-Aug 2017, but monthly revenue peaked in Nov 2017 and levelled off through 2018.
2. Only 3.00% of customers (2,801 of 93,350) bought more than once.
3. Late orders score 2.27 stars vs 4.29 on time. Orders 8+ days late drop to 1.70.
4. The top 3 categories (health_beauty, watches_gifts, bed_bath_table) make up 25.9% of revenue.
5. AL has the highest late rate (21.4%), but RJ has the most impact: 1,495 late orders (12.1%), 22.9% of all late orders.

## Recommendations
1. **Fix delivery where it affects the most orders.** Start with RJ and BA, then AL, MA, and SE. AL, MA, and SE have the worst rates but only 4.0% of late orders.
2. **Test a second-purchase offer on high-value one-time buyers.** Try a sample before a full rollout.
3. **Run a win-back message for lapsing repeat buyers.**

## Limits
Associations only, historical data, no marketing-cost data.